# E4B swing-task repeats
17 swing tasks from the Seed39 cohort, 100 independent attempts each, one frozen repair-decisions policy, 12 slots on each of four L4 lanes. No leaderboard submission.
Frozen notebook: rebuild with --rounds instead of editing cells.


In [ ]:

import os, subprocess
from pathlib import Path
os.environ.update(LITELLM_LOCAL_MODEL_COST_MAP="True", LITELLM_TELEMETRY="False",
                  VLLM_WORKER_MULTIPROC_METHOD="spawn", TRANSFORMERS_NO_TF="1",
                  VLLM_NO_USAGE_STATS="1", OTEL_SDK_DISABLED="true")
hardware = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                          capture_output=True, text=True, check=True).stdout.strip().splitlines()
print("ALLOCATED_GPUS", hardware, flush=True)
if len(hardware) != 4 or any("L4" not in name for name in hardware):
    raise RuntimeError(f"Registration requires exactly four L4s: {hardware}")


In [ ]:

# ---------- install the competition wheelhouse (adk_submission / swegemma / adk_eval_core) ----------
import glob, os, shutil, subprocess, sys
from pathlib import Path

print("listing /kaggle/input:", sorted(os.listdir("/kaggle/input")))
CANDIDATES = [
    Path("/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse"),  # layout observed 2026-10-30
    Path("/kaggle/input/gemma-4-developer-agent-wheelhouse"),                  # legacy flat layout
    Path("/kaggle/input/gemma-4-developer-agent-wheelhouse/datasets"),
]
WHEELHOUSE_DIR = next((p for p in CANDIDATES if p.is_dir()), None)
if WHEELHOUSE_DIR is None:
    for p in glob.glob("/kaggle/input/**/gemma-4-developer-agent-wheelhouse", recursive=True):
        WHEELHOUSE_DIR = Path(p)
        break
assert WHEELHOUSE_DIR is not None, "wheelhouse dataset not mounted"
print("WHEELHOUSE:", WHEELHOUSE_DIR)

for pat in ("/usr/local/lib/python*/dist-packages/*cutlass*.pth",
            "/usr/local/lib/python*/site-packages/*cutlass*.pth"):
    for pth in glob.glob(pat):
        try:
            os.unlink(pth)
        except OSError:
            pass

tmp_whl = Path("/tmp/wheelhouse")
if tmp_whl.exists():
    shutil.rmtree(tmp_whl)
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob("*.whl"):
    if "cutlass" in w.name.lower():
        continue
    name = w.name.replace("cu128", "+cu128") if ("cu128" in w.name and "+" not in w.name) else w.name
    target = tmp_whl / name
    if not target.exists():
        os.symlink(w, target)
wheels = sorted(str(w) for w in tmp_whl.glob("*.whl"))
names = [w.rsplit("/", 1)[-1] for w in wheels]
print(f"WHEELHOUSE globbed={len(wheels)} swegemma={any('swegemma' in n for n in names)}")
assert any("swegemma" in n for n in names), "wheelhouse lacks swegemma (wrong dataset version)"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                "--force-reinstall", *wheels], check=True)
import adk_submission, swegemma
print("IMPORTS OK: adk_submission", getattr(adk_submission, "__version__", "?"),
      "| swegemma", getattr(swegemma, "__version__", "?"))

if (adk_submission.__version__, swegemma.__version__) != ("0.2.13", "0.2.11"):
    raise RuntimeError("Registered ADK submission/evaluator versions were not mounted")


In [ ]:

import base64, hashlib, io, json, shutil, sys, tarfile
from pathlib import Path
from adk_submission import discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
from transformers import AutoTokenizer
WORK = Path("/kaggle/working")
bundle = base64.b64decode("...")
assert hashlib.sha256(bundle).hexdigest() == "416d76303425365732120608c20a3e5432a57e89f1dfd21720441057f5ac7924"
with tarfile.open(fileobj=io.BytesIO(bundle)) as archive:
    archive.extractall(WORK, filter="data")
TASKS = WORK / "tasks.jsonl"
TASK_IDS = json.loads((WORK / "ab_tasks.json").read_text())
assert len(TASK_IDS) == len(set(TASK_IDS)) == 39
matches = [p for p in Path("/kaggle/input").rglob("tasks.jsonl")
           if p.parent.name == "gemma129-taskdefs"]
assert len(matches) == 1, ("Registered task definition dataset missing or ambiguous", matches)
subset = {}
for line in matches[0].read_bytes().splitlines():
    if line.strip():
        task = json.loads(line)
        task_id = task["instance_id"]
        if task_id in TASK_IDS:
            assert task_id not in subset, ("Duplicate mounted task", task_id)
            subset[task_id] = task
assert set(subset) == set(TASK_IDS), "Mounted tasks missing registered IDs"
TASKS.write_bytes(b"".join((json.dumps(subset[task_id]) + "\n").encode() for task_id in TASK_IDS))
assert hashlib.sha256(TASKS.read_bytes()).hexdigest() == "b86f9beb9f924043362ea12c6398a8a961525dc10876bf6494d07f0591267ff9"
AGENT = WORK / "arms" / "G_startup_lora"
matches = [p for p in Path("/kaggle/input").rglob("adapter_model.safetensors")
           if p.parent.name == "e4b-repair-decisions-v1-pilot"]
assert len(matches) == 1, ("Registered adapter dataset missing or ambiguous", matches)
source = matches[0].parent
assert hashlib.sha256(matches[0].read_bytes()).hexdigest() == "9f6b42913234eb5b9ff27a9c686bb68348c29403ca0ba9c761c8d6538827cc85"
assert hashlib.sha256((source / "adapter_config.json").read_bytes()).hexdigest() == "a0d3ea9560655956305345562eeb7985a49f3622eff8e3539bf665343baa53cf"
destination = AGENT / "adapters" / "repair_decisions_v1"
destination.mkdir(parents=True, exist_ok=False)
for name in ("adapter_config.json", "adapter_model.safetensors"):
    shutil.copyfile(source / name, destination / name)
ADAPTERS = discover_adapters(str(AGENT), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
assert set(ADAPTERS.adapters) == {"repair_decisions_v1"}
SNAPS = None
for candidate in Path("/kaggle/input").rglob(TASK_IDS[0] + ".tgz"):
    if all((candidate.parent / (task + ".tgz")).is_file() for task in TASK_IDS):
        SNAPS = candidate.parent
        break
if SNAPS is None:
    archives = sorted(Path("/kaggle/input").rglob("snapshots.tar"))
    assert len(archives) == 1, ("Snapshots missing or ambiguous", archives)
    SNAPS = Path("/tmp/tool_lora_p5_snapshots")
    SNAPS.mkdir()
    with tarfile.open(archives[0]) as archive:
        archive.extractall(SNAPS, filter="data")
assert all((SNAPS / (task + ".tgz")).is_file() for task in TASK_IDS)
models = [p for p in Path("/kaggle/input/models/google/gemma-4").rglob("config.json")
          if p.parent.name == "2" and p.parent.parent.name == "gemma-4-e4b-it-qat-w4a16-ct"]
assert len(models) == 1, ("Exact registered E4B version2 model missing or ambiguous", models)
MODEL_PATH = models[0].parent
native_config = json.loads(models[0].read_text())
native_context = native_config.get("text_config", native_config).get("max_position_embeddings")
assert type(native_context) is int and native_context >= 32768, native_context
weights = [p for p in MODEL_PATH.glob("*.safetensors") if p.stat().st_size == 11513861828]
assert len(weights) == 1, ("Registered training-base weights missing", weights)
with weights[0].open("rb") as model_file:
    base_hash = hashlib.file_digest(model_file, "sha256").hexdigest()
assert base_hash == "19d89d2a4b3e401a3eae75ec08ab112810421a036c91d8c8226b686be7fe0c5d"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
CONTEXT_PROBE_TOKEN_ID = tokenizer.encode("hello", add_special_tokens=False)[0]
SMOKE_PROMPT = tokenizer.apply_chat_template(
    [{"role": "user", "content":
      "A source file was read and its defect is understood. Explain the next safe coding action."}],
    tokenize=False, add_generation_prompt=True, enable_thinking=False)
sys.path.insert(0, str(WORK / "support"))
(WORK / "input_receipt.json").write_text(json.dumps({
    "tasks": len(TASK_IDS), "bundle_sha256": "416d76303425365732120608c20a3e5432a57e89f1dfd21720441057f5ac7924",
    "native_context": native_context, "registered_context": 32768,
    "base_sha256": base_hash, "adapter_sha256": "9f6b42913234eb5b9ff27a9c686bb68348c29403ca0ba9c761c8d6538827cc85",
    "policy": "G_startup; one policy; 17 swing tasks x 100; 12 slots per lane",
}, indent=2))
print("REGISTERED_INPUTS_OK", len(TASK_IDS), base_hash, "9f6b42913234eb5b9ff27a9c686bb68348c29403ca0ba9c761c8d6538827cc85", flush=True)


In [ ]:
from kaggle_pkg.tool_lora_runner import run
run(WORK, AGENT, TASKS, TASK_IDS, SNAPS, MODEL_PATH, SMOKE_PROMPT, ADAPTERS,
    rounds=100, swing_repeat=True)
